# Feature Engineering

Feature engineering is the process of **transforming raw data into features** that better represent the underlying problem to predictive models, resulting in improved accuracy. In this notebook, we cover the most important techniques — all implemented in **pure Python**.

## 1. Introduction — What Is Feature Engineering?

Raw data is rarely in a form that machine learning algorithms can use directly. Feature engineering bridges this gap.

It includes:
- **Cleaning**: handling missing values, removing noise  
- **Encoding**: converting categorical data to numbers  
- **Scaling**: normalising feature ranges  
- **Selection**: choosing the most informative features  
- **Creation**: deriving new features from existing ones

### Why Does It Matter?

> *"Applied machine learning is basically feature engineering."* — Andrew Ng

A well-engineered feature set can:
- Improve model accuracy significantly  
- Reduce training time (fewer, better features)  
- Make simple models competitive with complex ones  
- Provide better interpretability

## 2. Handling Missing Values

Missing data is inevitable in real datasets. Common strategies:

| Strategy | When to Use |
|----------|-------------|
| **Mean** imputation | Numerical features with roughly symmetric distributions |
| **Median** imputation | Numerical features with skewed distributions or outliers |
| **Mode** imputation | Categorical features |
| **Drop** row/column | When very few values are missing, or a column is mostly empty |

In [ ]:
def mean_impute(column):
    """Replace None values with the column mean."""
    values = [v for v in column if v is not None]
    col_mean = sum(values) / len(values)
    return [v if v is not None else col_mean for v in column]


def median_impute(column):
    """Replace None values with the column median."""
    values = sorted(v for v in column if v is not None)
    n = len(values)
    if n % 2 == 1:
        col_median = values[n // 2]
    else:
        col_median = (values[n // 2 - 1] + values[n // 2]) / 2
    return [v if v is not None else col_median for v in column]


def mode_impute(column):
    """Replace None values with the most frequent value."""
    counts = {}
    for v in column:
        if v is not None:
            counts[v] = counts.get(v, 0) + 1
    col_mode = max(counts, key=counts.get)
    return [v if v is not None else col_mode for v in column]

In [ ]:
# --- Demonstration ---
ages = [25, 30, None, 22, 35, None, 28, 40, 22, None]
colors = ['red', 'blue', None, 'red', 'green', 'red', None, 'blue', 'red', 'red']

print("Original ages: ", ages)
print("Mean imputed:  ", mean_impute(ages))
print("Median imputed:", median_impute(ages))
print()
print("Original colors:", colors)
print("Mode imputed:   ", mode_impute(colors))

## 3. Label Encoding

**Label encoding** converts each unique category string to a unique integer. This is suitable for **ordinal** features (where the order matters, e.g. `low < medium < high`) or for algorithms that can handle integer-encoded categories (e.g. tree-based models).

**Warning**: For non-ordinal features, label encoding can introduce a false ordering that misleads distance-based or linear models. Use one-hot encoding instead.

In [ ]:
def label_encode(column):
    """
    Map each unique value in the column to an integer.

    Returns
    -------
    encoded : list of int
    mapping : dict {original_value: integer}
    """
    unique_values = sorted(set(column))
    mapping = {val: idx for idx, val in enumerate(unique_values)}
    encoded = [mapping[v] for v in column]
    return encoded, mapping

In [ ]:
sizes = ['small', 'medium', 'large', 'medium', 'small', 'large', 'large']
encoded_sizes, size_map = label_encode(sizes)

print("Original: ", sizes)
print("Encoded:  ", encoded_sizes)
print("Mapping:  ", size_map)

## 4. One-Hot Encoding

For **nominal** (unordered) categorical features, one-hot encoding creates a separate binary column for each category. This avoids imposing any ordinal relationship.

Example: `color ∈ {red, green, blue}` becomes three columns:

| color | is_blue | is_green | is_red |
|-------|---------|----------|--------|
| red   | 0       | 0        | 1      |
| green | 0       | 1        | 0      |
| blue  | 1       | 0        | 0      |

In [ ]:
def one_hot_encode(column):
    """
    One-hot encode a categorical column.

    Returns
    -------
    encoded : list of list of int  – each inner list is one row
    categories : list of str       – column names for the binary features
    """
    categories = sorted(set(column))
    cat_to_idx = {cat: i for i, cat in enumerate(categories)}
    n_cats = len(categories)

    encoded = []
    for value in column:
        row = [0] * n_cats
        row[cat_to_idx[value]] = 1
        encoded.append(row)

    return encoded, categories

In [ ]:
colors = ['red', 'green', 'blue', 'red', 'blue', 'green', 'red']
encoded_colors, color_cats = one_hot_encode(colors)

print(f"Categories: {color_cats}")
print("\nEncoded:")
for original, enc_row in zip(colors, encoded_colors):
    print(f"  {original:>6}  =>  {enc_row}")

## 5. Feature Scaling Comparison

Many algorithms (e.g. SVM, KNN, neural networks, gradient descent-based methods) are sensitive to the scale of features. Two common approaches:

### Min-Max Scaling (Normalisation)

Rescales values to the range [0, 1]:

$$x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

### Z-Score Standardisation

Transforms to zero mean and unit variance:

$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$

| Method | Range | Sensitive to Outliers? | Best For |
|--------|-------|------------------------|----------|
| Min-Max | [0, 1] | Yes | Bounded algorithms (e.g. neural nets with sigmoid) |
| Z-Score | ~ [-3, 3] | Less so | Algorithms assuming Gaussian-like distributions |

In [ ]:
import math


def min_max_scale(column):
    """Scale values to [0, 1]."""
    col_min = min(column)
    col_max = max(column)
    rng = col_max - col_min
    if rng == 0:
        return [0.0] * len(column)
    return [(v - col_min) / rng for v in column]


def z_score_scale(column):
    """Standardise to zero mean and unit variance."""
    n = len(column)
    mean = sum(column) / n
    variance = sum((v - mean) ** 2 for v in column) / n
    std = math.sqrt(variance)
    if std == 0:
        return [0.0] * len(column)
    return [(v - mean) / std for v in column]

In [ ]:
data = [10, 20, 30, 40, 50, 200]

print("Original:  ", data)
print("Min-Max:   ", [round(v, 4) for v in min_max_scale(data)])
print("Z-Score:   ", [round(v, 4) for v in z_score_scale(data)])
print()
print("Notice how the outlier (200) compresses the other Min-Max values")
print("toward 0, while Z-Score spreads them more evenly.")

## 6. Feature Selection

Not all features are useful. Irrelevant or redundant features can:
- Increase training time  
- Cause overfitting  
- Reduce model interpretability

### 6.1 Variance Threshold

Remove features whose variance falls below a threshold. Features with very low variance carry almost no information.

In [ ]:
def column_variance(column):
    """Compute the population variance of a column."""
    n = len(column)
    mean = sum(column) / n
    return sum((v - mean) ** 2 for v in column) / n


def variance_threshold_select(dataset, feature_names, threshold=0.01):
    """
    Select features whose variance exceeds the threshold.

    Parameters
    ----------
    dataset : list of list of float – rows are samples, columns are features
    feature_names : list of str
    threshold : float

    Returns
    -------
    filtered_data : list of list of float
    kept_names : list of str
    """
    n_features = len(dataset[0])
    keep_indices = []

    for j in range(n_features):
        col = [row[j] for row in dataset]
        var = column_variance(col)
        print(f"  Feature '{feature_names[j]}': variance = {var:.6f}"
              f"  {'KEEP' if var >= threshold else 'DROP'}")
        if var >= threshold:
            keep_indices.append(j)

    filtered_data = [[row[j] for j in keep_indices] for row in dataset]
    kept_names = [feature_names[j] for j in keep_indices]
    return filtered_data, kept_names

In [ ]:
sample_data = [
    [1.0, 0.0, 100],
    [2.0, 0.0, 200],
    [3.0, 0.0, 150],
    [4.0, 0.0, 300],
    [5.0, 0.0, 250],
]
names = ['height', 'constant', 'income']

print("Variance analysis:")
filtered, kept = variance_threshold_select(sample_data, names, threshold=0.1)
print(f"\nKept features: {kept}")
print(f"Filtered data: {filtered}")

### 6.2 Correlation-Based Selection

Highly correlated features provide redundant information. We can compute the **Pearson correlation** between each pair of features and drop one from pairs that exceed a correlation threshold.

In [ ]:
def pearson_correlation(x, y):
    """Compute Pearson correlation coefficient between two lists."""
    n = len(x)
    mean_x = sum(x) / n
    mean_y = sum(y) / n

    cov = sum((xi - mean_x) * (yi - mean_y) for xi, yi in zip(x, y)) / n
    std_x = math.sqrt(sum((xi - mean_x) ** 2 for xi in x) / n)
    std_y = math.sqrt(sum((yi - mean_y) ** 2 for yi in y) / n)

    if std_x == 0 or std_y == 0:
        return 0.0
    return cov / (std_x * std_y)


def correlation_matrix(dataset, feature_names):
    """Print a correlation matrix for the dataset."""
    n_features = len(feature_names)
    cols = [[row[j] for row in dataset] for j in range(n_features)]

    # Header
    header = f"{'':>12}" + "".join(f"{name:>12}" for name in feature_names)
    print(header)
    print("-" * len(header))

    for i in range(n_features):
        row_str = f"{feature_names[i]:>12}"
        for j in range(n_features):
            r = pearson_correlation(cols[i], cols[j])
            row_str += f"{r:>12.4f}"
        print(row_str)

In [ ]:
corr_data = [
    [1.0, 2.1, 10],
    [2.0, 4.0, 20],
    [3.0, 5.9, 15],
    [4.0, 8.1, 25],
    [5.0, 9.8, 30],
]
corr_names = ['x1', 'x2_almost_2x1', 'x3_different']

print("Correlation matrix:")
correlation_matrix(corr_data, corr_names)
print("\nNote: x1 and x2 are nearly perfectly correlated (≈1.0).")
print("We could drop one of them without losing much information.")

## 7. Feature Creation

Sometimes the raw features don't capture the relationships the model needs. We can **create new features** that do.

### 7.1 Polynomial Features

For features $[x_1, x_2]$, degree-2 polynomial features produce:

$$[x_1, x_2, x_1^2, x_1 x_2, x_2^2]$$

This lets a linear model learn non-linear patterns.

### 7.2 Interaction Terms

Interaction features (products of feature pairs) capture combined effects that neither feature captures alone.

In [ ]:
def polynomial_features(dataset, degree=2):
    """
    Generate polynomial features up to the given degree.

    For simplicity, handles degree=2 with original features,
    squared terms, and pairwise interaction terms.
    """
    result = []
    for row in dataset:
        new_row = list(row)  # original features
        n = len(row)

        if degree >= 2:
            # Squared terms
            for i in range(n):
                new_row.append(row[i] ** 2)
            # Interaction terms
            for i in range(n):
                for j in range(i + 1, n):
                    new_row.append(row[i] * row[j])

        result.append(new_row)
    return result


def interaction_features(dataset):
    """Create only the pairwise interaction terms (no squared terms)."""
    result = []
    for row in dataset:
        new_row = list(row)
        n = len(row)
        for i in range(n):
            for j in range(i + 1, n):
                new_row.append(row[i] * row[j])
        result.append(new_row)
    return result

In [ ]:
original = [
    [2, 3],
    [1, 4],
    [5, 2],
]
feature_names_orig = ['x1', 'x2']
poly_names = ['x1', 'x2', 'x1^2', 'x2^2', 'x1*x2']

poly_data = polynomial_features(original, degree=2)

print("Original features:", feature_names_orig)
print("Polynomial features:", poly_names)
print()
for orig_row, poly_row in zip(original, poly_data):
    print(f"  {orig_row}  =>  {poly_row}")

In [ ]:
interact_data = interaction_features(original)
interact_names = ['x1', 'x2', 'x1*x2']

print("Interaction features only:", interact_names)
for orig_row, inter_row in zip(original, interact_data):
    print(f"  {orig_row}  =>  {inter_row}")

### Complete Example: Feature Engineering Pipeline

Let's put it all together on a small mixed-type dataset.

In [ ]:
# Raw dataset: each row is [age, income, city, purchased]
raw_data = [
    [25,   50000,  'New York',  'yes'],
    [30,   60000,  'London',    'no'],
    [None, 80000,  'Paris',     'yes'],
    [35,   None,   'London',    'yes'],
    [40,   70000,  'New York',  'no'],
    [28,   55000,  'Paris',     'yes'],
    [None, 90000,  'New York',  'no'],
    [33,   65000,  'London',    'yes'],
]

print("=== Step 1: Handle missing values ===")
ages = [row[0] for row in raw_data]
incomes = [row[1] for row in raw_data]
ages_filled = median_impute(ages)
incomes_filled = mean_impute(incomes)
print(f"Ages (median imputed):   {ages_filled}")
print(f"Incomes (mean imputed):  {incomes_filled}")

print("\n=== Step 2: Encode categorical features ===")
cities = [row[2] for row in raw_data]
city_encoded, city_cats = one_hot_encode(cities)
print(f"City categories: {city_cats}")
for city, enc in zip(cities, city_encoded):
    print(f"  {city:>10} => {enc}")

# Encode target
purchased = [row[3] for row in raw_data]
target_encoded, target_map = label_encode(purchased)
print(f"\nTarget mapping: {target_map}")
print(f"Encoded target: {target_encoded}")

print("\n=== Step 3: Scale numerical features ===")
ages_scaled = z_score_scale(ages_filled)
incomes_scaled = min_max_scale(incomes_filled)
print(f"Ages (z-score):     {[round(v, 3) for v in ages_scaled]}")
print(f"Incomes (min-max):  {[round(v, 3) for v in incomes_scaled]}")

print("\n=== Step 4: Assemble final feature matrix ===")
X_final = []
for i in range(len(raw_data)):
    row = [ages_scaled[i], incomes_scaled[i]] + city_encoded[i]
    X_final.append(row)

final_names = ['age_z', 'income_mm'] + [f'city_{c}' for c in city_cats]
print(f"Feature names: {final_names}")
for row in X_final:
    print(f"  {[round(v, 3) for v in row]}")

## 8. Summary — Best Practices

### Dos

1. **Always handle missing values** before modelling. Choose the strategy based on the distribution and feature type.  
2. **Encode categoricals** appropriately: use **label encoding** for ordinal features, **one-hot encoding** for nominal features.  
3. **Scale features** when using distance-based or gradient-based algorithms. Fit the scaler on training data only, then apply to test data.  
4. **Remove low-variance** features — they add noise without information.  
5. **Check correlations** — highly correlated features are redundant and can hurt regularised models.  
6. **Create polynomial/interaction features** when you suspect non-linear relationships.

### Don'ts

1. Don't use label encoding for nominal features with linear models — it introduces a false ordinal relationship.  
2. Don't scale the target variable in classification tasks.  
3. Don't fit the scaler or imputer on the test set — this causes **data leakage**.  
4. Don't blindly create all possible polynomial features — the number grows combinatorially and can lead to overfitting.  
5. Don't ignore **domain knowledge** — the best features often come from understanding the problem, not from automated methods.

### Quick Reference

| Task | Method | When |
|------|--------|------|
| Missing numerical | Mean / Median imputation | Symmetric / Skewed distribution |
| Missing categorical | Mode imputation | Any categorical feature |
| Ordinal categories | Label encoding | Natural order exists |
| Nominal categories | One-hot encoding | No natural order |
| Feature scaling | Min-Max / Z-Score | Distance-based / Gradient-based models |
| Remove useless features | Variance threshold | Low-variance columns |
| Remove redundant features | Correlation analysis | Highly correlated pairs |
| Capture non-linearity | Polynomial features | Suspected non-linear relationships |